## 1 Fluxo de _chats_

Vamos introduzir um novo fluxo de interação com o LangChain, saindo do formato de execução única para a simulação de uma conversa de _chat_. Para isso vamos usar a classe [ChatOpenAI](https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI) para fazer a interface com a API para _chats_ da OpenAI.

Em seguida, criamos uma lista de perguntas sequenciais e utilizamos um _loop_ `for` para enviar cada uma dessas perguntas em uma requisição ao modelo.

In [29]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
api_key       = os.getenv("OPENAI_API_KEY")
model_version = os.getenv("OPENAI_MODEL_VERSION")

llm = ChatOpenAI(
    model=model_version,
    api_key=api_key
)

messages = [
    "Quero visitar um lugar no Brasil, famoso por praias e cultura. Pode sugerir?",
    "Qual a melhor época do ano para visitar esse lugar?"
]

for query in messages:
    response = llm.invoke(query)
    print("- User:", query)
    print("- IA:", response.content, "\n")

- User: Quero visitar um lugar no Brasil, famoso por praias e cultura. Pode sugerir?
- IA: **Salvador, na Bahia**, combina muito bem praias e cultura. Você pode passear pelo Pelourinho e conhecer a música, a culinária e a história afro-brasileira; depois, relaxar em praias como Porto da Barra e Itapuã.

Se preferir, também posso sugerir um destino mais tranquilo, mais badalado ou com natureza. 

- User: Qual a melhor época do ano para visitar esse lugar?
- IA: Qual lugar você tem em mente? Diga o destino e eu indico a melhor época para visitar. 



Note que o modelo não mantém o contexto ou a memória das interações anteriores. Por padrão, o contexto não é acumulado ao longo de interações como essas que fizemos. Será necessário adicionar uma camada ao processo que faça a gestão do histórico da conversa.

Para isso vamos criar um _template_ de _chat_, começando pelo _import_ de [`ChatPromptTemplate`](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate), que é uma classe usada para estruturar e gerar sequências de mensagens formatadas para modelos de chat. E então criamos o _template_ "construido a partir de mensagens". As mensagens são passadas em uma lista, e cada uma delas será composta, além de seu conteúdo, também pelo _role_ de quem está lendo a mensagem.

Os _roles_ representam as entidades que compõem o diálogo na arquitetura de LLMs. No LangChain, os principais _roles_ utilizados são:

- `system`: define as instruções, o comportamento e a personalidade da IA. Esta mensagem geralmente guia o tom de toda a interação.
- `human`: representa as entradas ou perguntas feitas pelo usuário final
- `ai`: representa as respostas geradas pelo modelo. Quando implementamos um histórico, essas mensagens são inseridas ali para que o modelo saiba o que ele mesmo já respondeu anteriormente.
- `placeholder`: funciona como um compartimento vazio, que é preenchido por uma lista contendo esses papéis (`human` e `ai`) intercalados, formando o histórico da conversa.
- `tool`: utilizado para representar o resultado de alguma ferramenta que a IA executou.

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_destination = ChatPromptTemplate.from_messages([
    ("system",      "Você é um guia de viagem especializado em destinos brasileiros. Apresente-se como Sr. Passeios"),
    ("placeholder", "{conversation_historic}"),
    ("human",       "{query}")
])

chain = prompt_destination | llm | StrOutputParser()

A cadeia definida acima serve ao propósito de estruturar como a informação flui entre o _prompt_, o modelo e o _parser_. Ou seja, a cadeia é o "caminho" que a mensagem percorre.

Agora vamos implementar a gestão da memória no _chat_. Precisamos de um contentor para armazenar todas as trocas de mensagens que acontecerem, e este será o dicionário `memory`, pois vamos manter as conversas apenas na memória RAM. Para identificar uma conversa vamos definir seu identificador em `session_id` e para recuperar esse histórico definimos a função `conversarion_per_session`:

In [31]:
from langchain_core.chat_history import InMemoryChatMessageHistory

memory     = {}
session_id = "travel_plan" # adicionar um timestmap

def conversarion_per_session(session_id: str):
    if session_id not in memory:
        memory[session_id] = InMemoryChatMessageHistory()
    
    return memory[session_id]

Note que a função `conversarion_per_session` segue o padrão de projeto _singleton_, que tem como objetivo garantir que a mesma instância de histórico seja sempre acessada para uma mesma chave de identificação.

A função verifica se já existe uma memória com o identificador passado:

- se a resposta for **não**: é criado um novo registro (uma instância de [`InMemoryChatMessageHistory`](https://reference.langchain.com/python/langchain-core/chat_history/InMemoryChatMessageHistory)) e adicionado no dicionário `memory`. A classe `InMemoryChatMessageHistory` é usada para armazenar o histórico de mensagens de uma conversa diretamente na memória RAM da aplicação. Ou seja, o dicionário `memory` contém os endereços lógicos de memória onde o histórico das conversas estão armazenados; 
- se a resposta for **sim**: simplesmente retorna o endereço de memória com o registro já criado anteriormente, ou seja, o acesso direto ao mesmo objeto.


Isso é importante porque evita criar múltiplos objetos de histórico para a mesma sessão, se não usássemos essa lógica, toda vez que o código rodasse poderíamos acidentalmente criar uma nova "memória" vazia, perdendo o que foi conversado anteriormente (eficiência). E como queremos que o _chat_ se lembre do que foi dito nas mensagens passadas, precisamos garantir que a cadeia com memória sempre aponta-rá para o mesmo objeto onde essas mensagens estão sendo armazenadas (persitência de estado).

Agora podemos criar a "cadeia com memória", definindo sua execução com um _runnable_, a classe [`RunnableWithMessageHistory`](https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory), que gerencia o histórico de mensagens para outro _runnable_, no caso a cadeia definida acima. 

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_memory = RunnableWithMessageHistory(
    runnable=chain,                                 # runnable base para o qual sera gerenciado o historico de mensagens
    get_session_history=conversarion_per_session,   # funcao que retorna um historico de mensagens a partir de um session_id
    input_messages_key="query",                     # inputs definidos no runnable base
    history_messages_key="conversation_historic"    # placeholder definido no runnable base
)

/home/bruno/estudos/ai-engineering-portifolio/travel-assistant-agent/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Note que, como o `RunnableWithMessageHistory` usa o endereço de memória para acessar o objeto com o histórico de mensagens e salvar uma nova, ele está alterando o conteúdo do objeto original. Como o dicionário `memory` aponta para esse mesmo lugar, na próxima consulta, o histórico já estará atualizado com as últimas mensagens. Como estamos tratando de um objeto que precisa ser mutável (que vai crescer conforme a conversa avança), mantemos a referência a ele para que possamos "escrever" novas mensagens dentro dele durante toda a sessão.

Enfim, modificamos o _loop_ para a resposta de cada iteração (mensagem/pergunta) seja a execução da cadeia com memória. Passamos para o método `invoke`: a pergunta e a configuração que define qual será o histórico utilizado, cada um em um dicionário:

In [33]:
for query in messages:
    response = chain_with_memory.invoke(
        {"query": query},
        {"configurable": {"session_id": session_id}}
    )
    
    print("- User:", query)
    print("- IA:", response, "\n") # lembrando que a cadeia tem um output parser que transforma a resposta em string

- User: Quero visitar um lugar no Brasil, famoso por praias e cultura. Pode sugerir?
- IA: Olá! Sou o **Sr. Passeios**, seu guia de viagem pelo Brasil. Minha sugestão é **Salvador, na Bahia**: você encontra praias como Porto da Barra e Flamengo, além de uma cultura marcante, com o Pelourinho, o acarajé, a música e a arquitetura histórica.  
 

- User: Qual a melhor época do ano para visitar esse lugar?
- IA: Para visitar **Salvador**, eu recomendaria **setembro a novembro**: costuma fazer calor, há boas condições para curtir as praias e a cidade tende a estar menos cheia do que no verão.

**De dezembro a março** também é uma ótima época, com clima quente e muita animação, mas há mais turistas e os preços podem subir — especialmente no Carnaval. Já **abril a julho** costuma ser um período mais chuvoso.

— **Sr. Passeios** 

